In [ ]:
!pip install -q transformers yfinance

import os, zipfile, warnings, io, sys
import numpy as np
import torch
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import yfinance as yf
warnings.filterwarnings('ignore')

print('✅ Ready')

✅ Ready


In [ ]:
from google.colab import files

print('📂 Upload emotion_finance_model.zip ...')
uploaded = files.upload()

MODEL_DIR = './BEST_MODEL_FINAL.zip'

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('.')
        print(f'✅ Extracted {fname}')

# Auto-detect model folder (handles nested zips)
if not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    for root, dirs, fnames in os.walk('.'):
        if 'config.json' in fnames:
            MODEL_DIR = root
            break

print(f'📁 Model folder : {MODEL_DIR}')
print(f'   Files        : {os.listdir(MODEL_DIR)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, torch_dtype=torch.float32)
model.eval()
id2label  = model.config.id2label

print(f'\n✅ Model loaded  |  labels: {id2label}')

📂 Upload emotion_finance_model.zip ...


Saving BEST_MODEL_FINAL.zip to BEST_MODEL_FINAL.zip


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Extracted BEST_MODEL_FINAL.zip
📁 Model folder : .
   Files        : ['.config', 'model.safetensors', 'tokenizer.json', 'config.json', 'BEST_MODEL_FINAL.zip', 'tokenizer_config.json', 'sample_data']


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


✅ Model loaded  |  labels: {0: 'stressed', 1: 'neutral', 2: 'confident'}


In [ ]:
EXPENSE_COLS = [
    'Rent','Loan_Repayment','Insurance','Groceries','Transport',
    'Eating_Out','Entertainment','Utilities','Healthcare','Education','Miscellaneous'
]

STRATEGIES = {
    'stressed': {
        'risk':'LOW', 'icon':'🔴',
        'instruments':['Liquid Mutual Fund','FD (short-term)','PPF','Sukanya / NSC'],
        'actions':[
            'Pause all non-essential investments immediately',
            'Build 3-month emergency fund before anything else',
            'Prepay highest-interest loan first',
            'Cut Eating_Out + Entertainment by 50%',
            'Auto-transfer savings on salary day'
        ],
        'alloc':[('Emergency Fund top-up',0.70),('Loan Prepayment',0.20),('Liquid Fund',0.10)]
    },
    'neutral': {
        'risk':'MODERATE', 'icon':'🟡',
        'instruments':['Nifty 50 Index Fund','Balanced Advantage Fund','ELSS (tax saving)','Short-term Debt Fund'],
        'actions':[
            'Start a monthly SIP — consistency beats timing',
            'Keep 6-month emergency fund in liquid fund',
            'Max out Section 80C (₹1.5L/yr) via ELSS',
            'Top up term insurance if under-insured',
            'Automate SIP — never invest manually'
        ],
        'alloc':[('Index Fund SIP',0.50),('ELSS / Tax Saving',0.25),('Debt / Liquid Fund',0.25)]
    },
    'confident': {
        'risk':'HIGH', 'icon':'🟢',
        'instruments':['Direct Equity / Bluechip','Flexi-cap Mutual Fund','Mid/Small-cap SIP','REITs / InvITs'],
        'actions':[
            'Deploy full investable amount across equity + SIP',
            'Diversify: 60% large-cap, 30% mid-cap, 10% international',
            'Increase SIP by 10% every year',
            'Explore NPS Tier-1 for extra tax benefit',
            'Set target corpus and review allocation every 6 months'
        ],
        'alloc':[('Direct Equity / Large-cap',0.40),('Mid/Small-cap SIP',0.30),('International Fund',0.10),('Debt / Safety Net',0.20)]
    }
}

# ── Metrics ──────────────────────────────────────────────────
def compute_metrics(profile):
    income    = float(profile.get('Income', 0))
    total_exp = sum(float(profile.get(c, 0)) for c in EXPENSE_COLS)
    savings   = income - total_exp
    sr        = savings / income if income > 0 else 0
    dti       = float(profile.get('Loan_Repayment', 0)) / income if income > 0 else 0
    emergency = float(profile.get('Emergency_Fund', 0))
    emg_need  = total_exp * 6
    emg_cov   = (emergency / emg_need * 100) if emg_need > 0 else 0
    emg_months= emergency / total_exp if total_exp > 0 else 0
    return dict(income=income, total_exp=total_exp, savings=savings,
                sr=sr, dti=dti, emergency=emergency,
                emg_need=emg_need, emg_cov=emg_cov, emg_months=emg_months)

# ── Model inference ──────────────────────────────────────────
def predict_emotion(profile, m):
    emi  = float(profile.get('Loan_Repayment', 0))
    text = (
        f"Monthly income of ₹{m['income']:.0f} with expenses of ₹{m['total_exp']:.0f} "
        f"leaving ₹{m['savings']:.0f} savings ({m['sr']*100:.1f}%). "
        f"Loan EMI ₹{emi:.0f}. Emergency fund covers {m['emg_months']:.1f} months."
    )
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       padding='max_length', max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs  = softmax(logits, dim=1)[0].numpy()
    idx    = int(probs.argmax())
    label  = id2label[idx]
    return label, float(probs[idx]), {id2label[i]: float(probs[i]) for i in range(len(probs))}, text

# ── Investable amount ────────────────────────────────────────
def compute_investable(profile, m, emotion):
    risk_map = {'stressed': 0.3, 'neutral': 0.6, 'confident': 0.8}
    if m['savings'] <= 0:
        return 0.0
    base = 0.5 if m['emergency'] < m['total_exp'] * 3 else 0.8
    return m['savings'] * base * risk_map[emotion]

# ── Spending alerts ───────────────────────────────────────────
def spending_alerts(profile, m):
    inc    = m['income']
    alerts = []
    checks = {
        'Rent':           (0.30, '🏠 Rent is {:.1f}% of income (>30%) — consider cheaper accommodation'),
        'Loan_Repayment': (0.20, '💳 Loan EMI is {:.1f}% of income (>20%) — prioritise prepayment'),
        'Eating_Out':     (0.08, '🍔 Eating out is {:.1f}% of income (>8%) — meal prep can help'),
        'Entertainment':  (0.05, '🎮 Entertainment is {:.1f}% of income (>5%) — review subscriptions'),
    }
    for col, (thresh, msg) in checks.items():
        val = float(profile.get(col, 0))
        pct = val / inc * 100 if inc > 0 else 0
        if pct > thresh * 100:
            alerts.append(msg.format(pct))
    if m['emg_months'] < 3:
        alerts.append(f'🚨 Emergency fund covers only {m["emg_months"]:.1f} months — target 6 months')
    return alerts

# ── Market analysis (yFinance — free) ────────────────────────
def analyse_stock(ticker):
    try:
        df    = yf.download(ticker, period='3mo', progress=False, auto_adjust=True)
        if df.empty:
            return None
        close = df['Close'].squeeze()
        last  = float(close.iloc[-1])
        prev  = float(close.iloc[-2])
        ma20  = float(close.rolling(20).mean().iloc[-1])
        ma50  = float(close.rolling(50).mean().iloc[-1])
        m1ago = float(close.iloc[-21]) if len(close) > 21 else float(close.iloc[0])
        vol   = float(close.pct_change().dropna().std() * (252**0.5) * 100)
        return dict(ticker=ticker, last=last,
                    pct_1d=(last-prev)/prev*100,
                    pct_1m=(last-m1ago)/m1ago*100,
                    ma20=ma20, ma50=ma50,
                    trend='bullish' if ma20 > ma50 else 'bearish',
                    above_ma20=last > ma20,
                    volatility=vol)
    except Exception as ex:
        print(f'  ⚠️  {ticker}: {ex}')
        return None

def stock_signal(s, emotion, investable, n_stocks):
    if s is None:
        return 'HOLD', 'Could not fetch market data.', 0, 0.0
    per_stock = investable / max(n_stocks, 1)
    if emotion == 'stressed':
        return 'AVOID', 'Resolve financial stress before investing.', 0, 0.0
    if s['trend'] == 'bullish' and s['above_ma20']:
        if emotion == 'confident':
            qty = int(per_stock // s['last'])
            return 'BUY', f"Confirmed uptrend. MA20 ₹{s['ma20']:.2f} > MA50 ₹{s['ma50']:.2f}. Price above MA20.", qty, qty * s['last']
        else:
            sip = per_stock
            return 'SIP', f"Bullish trend but moderate profile — start SIP of ₹{sip:.0f}/month.", 0, sip
    if s['trend'] == 'bullish' and not s['above_ma20']:
        sip = per_stock
        return 'SIP', f"Long-term uptrend but price below MA20 — good SIP entry. ₹{sip:.0f}/month.", 0, sip
    if s['trend'] == 'bearish' and s['above_ma20']:
        return 'HOLD', f"Bearish phase (MA20 < MA50). Wait for crossover before buying.", 0, 0.0
    return 'AVOID', f"Downtrend confirmed. Price below MA20 and MA50. Stay out.", 0, 0.0

# ── Master plan function ──────────────────────────────────────
def generate_plan(profile, stock_tickers=None):
    m          = compute_metrics(profile)
    emotion, conf, all_probs, text = predict_emotion(profile, m)
    investable = compute_investable(profile, m, emotion)
    strat      = STRATEGIES[emotion]
    alerts     = spending_alerts(profile, m)
    n          = len(stock_tickers) if stock_tickers else 1

    stock_results = []
    for t in (stock_tickers or []):
        print(f'  📡 Fetching {t} ...')
        s = analyse_stock(t)
        action, reason, qty, deployed = stock_signal(s, emotion, investable, n)
        stock_results.append(dict(ticker=t, market=s, action=action,
                                  reason=reason, qty=qty, deployed=deployed))

    return dict(emotion=emotion, confidence=conf, all_probs=all_probs,
                investable=investable, metrics=m, strategy=strat,
                alerts=alerts, stocks=stock_results, input_text=text)

# ── Report printer ────────────────────────────────────────────
def print_report(plan):
    s   = plan['strategy']
    m   = plan['metrics']
    iv  = plan['investable']
    SEP = '═' * 65
    sep = '─' * 65

    print(f'\n{SEP}')
    print(f'  💸 FINSENSE — INVESTMENT ACTION REPORT')
    print(SEP)

    print(f'\n{s["icon"]}  EMOTIONAL STATE : {plan["emotion"].upper()}  (confidence {plan["confidence"]:.1%})')
    print(f'    Risk Profile  : {s["risk"]}')
    print(f'    Input text    : "{plan["input_text"]}"')
    print(f'\n    Probability breakdown:')
    for lbl, p in plan['all_probs'].items():
        bar = '█' * int(p * 30)
        print(f'      {lbl:12} {bar:<32} {p:.1%}')

    print(f'\n{sep}')
    print('  📊 FINANCIAL SNAPSHOT')
    print(sep)
    print(f'  Monthly Income      : ₹{m["income"]:>10,.0f}')
    print(f'  Total Expenses      : ₹{m["total_exp"]:>10,.0f}')
    print(f'  Monthly Savings     : ₹{m["savings"]:>10,.0f}')
    print(f'  Savings Rate        : {m["sr"]*100:>10.1f}%')
    print(f'  Debt-to-Income      : {m["dti"]*100:>10.1f}%')
    print(f'  Emergency Fund      : ₹{m["emergency"]:>10,.0f}  ({m["emg_months"]:.1f} months)')
    print(f'  ── Investable/month : ₹{iv:>10,.0f}')

    if plan['alerts']:
        print(f'\n{sep}')
        print('  ⚠️  SPENDING ALERTS')
        print(sep)
        for a in plan['alerts']:
            print(f'  {a}')

    if plan['stocks']:
        print(f'\n{sep}')
        print('  📈 STOCK / FUND ACTIONS')
        print(sep)
        icons = {'BUY':'🟢','SIP':'🔵','HOLD':'🟡','AVOID':'🔴'}
        for sa in plan['stocks']:
            mk = sa['market']
            print(f'\n  {icons.get(sa["action"],"⚪")} [{sa["action"]}]  {sa["ticker"]}')
            if mk:
                print(f'     Price     : ₹{mk["last"]:.2f}  ({mk["pct_1d"]:+.2f}% today, {mk["pct_1m"]:+.2f}% 1-month)')
                print(f'     MA20/MA50 : ₹{mk["ma20"]:.2f} / ₹{mk["ma50"]:.2f}  → {mk["trend"].upper()}')
                print(f'     Volatility: {mk["volatility"]:.1f}% annualised')
            print(f'     Reason    : {sa["reason"]}')
            if sa['action'] == 'BUY' and sa['qty'] > 0:
                print(f'     ✅ EXECUTE  : BUY {sa["qty"]} shares @ ₹{mk["last"]:.2f} = ₹{sa["deployed"]:.2f}')
            elif sa['action'] == 'SIP':
                print(f'     ✅ EXECUTE  : SIP ₹{sa["deployed"]:.0f}/month in {sa["ticker"]}')

    print(f'\n{sep}')
    print(f'  🎯 PRIORITY ACTION PLAN  ({plan["emotion"].upper()})')
    print(sep)
    for i, a in enumerate(s['actions'], 1):
        print(f'  {i}. {a}')

    print(f'\n{sep}')
    print(f'  💼 RECOMMENDED INSTRUMENTS  (risk: {s["risk"]})')
    print(sep)
    for inst in s['instruments']:
        print(f'  • {inst}')

    print(f'\n{sep}')
    print('  💰 MONTHLY ALLOCATION')
    print(sep)
    for name, pct in s['alloc']:
        bar = '█' * int(pct * 30)
        print(f'  {name:<32} {bar:<32} ₹{iv*pct:,.0f}  ({pct:.0%})')

    print(f'\n{SEP}')
    print(f'  ✅ DONE  |  Investable: ₹{iv:,.0f}/month')
    print(SEP)

print('✅ All functions loaded — proceed to Cell 4')

✅ All functions loaded — proceed to Cell 4


In [ ]:
!pip install kiteconnect

In [ ]:
from kiteconnect import KiteConnect

API_KEY = "your_api_key"
API_SECRET = "your_api_secret"
ACCESS_TOKEN = "your_access_token"   # generated after login flow

kite = KiteConnect(api_key=API_KEY)
kite.set_access_token(ACCESS_TOKEN)

print("✅ Broker API connected")

In [ ]:
LIVE_TRADING = False   # always False for demo

def execute_trade(action, ticker, qty, price=None):
    """
    Broker execution layer (SAFE DEMO MODE)
    """

    order = {
        "action": action,
        "ticker": ticker,
        "quantity": qty,
        "price": price,
        "status": "SIMULATED"
    }

    if LIVE_TRADING:
        # 👉 REAL API CALL GOES HERE (Zerodha / Upstox)
        print(f"🚀 LIVE ORDER SENT: {order}")
    else:
        # 🧪 DEMO MODE (what you're using now)
        print(f"🧪 PAPER TRADE EXECUTED: {order}")

    return order

In [ ]:
for sa in plan['stocks']:
    mk = sa['market']

    print(f'\n  [{sa["action"]}]  {sa["ticker"]}')

    if sa['action'] == 'BUY' and sa['qty'] > 0:
        result = execute_trade("BUY", sa["ticker"], sa["qty"], mk["last"])
        print(f"     🧪 RESULT: {result['status']} ORDER")

    elif sa['action'] == 'SIP':
        print(f"     🧪 SIMULATED SIP: ₹{sa['deployed']:.0f}/month")

    elif sa['action'] == 'HOLD':
        print("     🟡 HOLD — no action")

    elif sa['action'] == 'AVOID':
        print("     🔴 AVOID — no action")

In [ ]:
import time

def execute_trade(action, ticker, qty, price=None):
    time.sleep(1)  # simulate broker latency

    order = {
        "action": action,
        "ticker": ticker,
        "quantity": qty,
        "price": price,
        "status": "FILLED (SIMULATED)"
    }

    print(f"🧠 Agent executed: {order}")
    return order

In [ ]:
MY_PROFILE = {
    # ── Income ──────────────────────────────────
    'Income':           80000,

    # ── Fixed ───────────────────────────────────
    'Rent':             20000,
    'Loan_Repayment':    5000,
    'Insurance':         3000,
    'Utilities':         3500,

    # ── Variable ────────────────────────────────
    'Groceries':         8000,
    'Transport':         4000,
    'Eating_Out':        3000,
    'Entertainment':     2000,
    'Healthcare':        2000,
    'Education':            0,
    'Miscellaneous':     1500,

    # ── Savings buffer ──────────────────────────
    'Emergency_Fund':  100000,
}

# NSE tickers end in .NS  |  BSE tickers end in .BO
# ETFs: 'NIFTYBEES.NS' (Nifty50), 'JUNIORBEES.NS' (Nifty Next50)
MY_STOCKS = [
    'RELIANCE.NS',
    'TCS.NS',
    'NIFTYBEES.NS',
]

# ── Run ─────────────────────────────────────────
print('🔄 Generating plan ...\n')
plan = generate_plan(MY_PROFILE, stock_tickers=MY_STOCKS)
print_report(plan)

🔄 Generating plan ...

  📡 Fetching RELIANCE.NS ...
  📡 Fetching TCS.NS ...
  📡 Fetching NIFTYBEES.NS ...

═════════════════════════════════════════════════════════════════
  💸 FINSENSE — INVESTMENT ACTION REPORT
═════════════════════════════════════════════════════════════════

🟡  EMOTIONAL STATE : NEUTRAL  (confidence 96.5%)
    Risk Profile  : MODERATE
    Input text    : "Monthly income of ₹80000 with expenses of ₹52000 leaving ₹28000 savings (35.0%). Loan EMI ₹5000. Emergency fund covers 1.9 months."

    Probability breakdown:
      stressed                                      2.0%
      neutral      ████████████████████████████     96.5%
      confident                                     1.4%

─────────────────────────────────────────────────────────────────
  📊 FINANCIAL SNAPSHOT
─────────────────────────────────────────────────────────────────
  Monthly Income      : ₹    80,000
  Total Expenses      : ₹    52,000
  Monthly Savings     : ₹    28,000
  Savings Rate        :  

In [ ]:
buf = io.StringIO()
sys.stdout = buf
print_report(plan)
sys.stdout = sys.__stdout__

with open('investment_report.txt', 'w') as f:
    f.write(buf.getvalue())

from google.colab import files
files.download('investment_report.txt')
print('✅ Downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>